# Numerical Rank of the GPT-2 Covariance Operator

This notebook reproduces the experiment reported in Table 1. For pretrained GPT-2 small, it computes

$C_\pi^i=\sum_j a_j^i(v^j-\bar v^i)(k^j-\bar k^i)^\top$

at selected query positions for every layer and attention head, and evaluates its numerical rank against the rank bound in Corollary M.3.


In [ ]:
# Keep the pandas and NumPy versions preinstalled in Colab.
%pip install -q --upgrade-strategy only-if-needed \
    "transformers==5.14.1" \
    "datasets>=3,<5"


In [ ]:
import math
import random
import shutil
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

SEED = 0
MODEL_NAME = "openai-community/gpt2"
DATASET_NAME = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
DATASET_SPLIT = "test"

SEQ_LEN = 128
N_BLOCKS = 10

# 0-based query positions.
# The corresponding numbers of visible tokens are 2, 4, 8, 16, 32, 64, and 128.
QUERY_POSITIONS = [1, 3, 7, 15, 31, 63, 127]

RESULT_DIR = Path("/content/gpt2_covariance")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert max(QUERY_POSITIONS) < SEQ_LEN

print("device:", device)
print("torch:", torch.__version__)


## 1. Load Pretrained GPT-2 Small

No training is performed. Hidden states entering each attention module are captured with hooks and used to reconstruct the corresponding $Q$, $K$, $V$, and causal softmax attention weights.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

model.config.use_cache = False
model.eval()
model.to(device)

n_layers = int(model.config.n_layer)
n_heads = int(model.config.n_head)
hidden_size = int(model.config.n_embd)
head_dim = hidden_size // n_heads
parameter_count = sum(parameter.numel() for parameter in model.parameters())

print("model:", MODEL_NAME)
print("parameters:", f"{parameter_count:,}")
print("layers:", n_layers)
print("heads per layer:", n_heads)
print("head dimension:", head_dim)

captured_attention_inputs = {}

def make_attention_input_hook(layer_index):
    def hook(module, args):
        if not args:
            raise RuntimeError("attention module received no positional input")
        captured_attention_inputs[layer_index] = args[0].detach()
    return hook

hook_handles = [
    block.attn.register_forward_pre_hook(
        make_attention_input_hook(layer_index)
    )
    for layer_index, block in enumerate(model.h)
]


## 2. Construct Fixed-Length Sequences from WikiText-2

Tokenize the WikiText-2 test split and construct 10 sequences of length 128.


In [ ]:
dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=DATASET_SPLIT,
)

required_tokens = N_BLOCKS * SEQ_LEN
token_ids = []

for row in dataset:
    text = row["text"].strip()
    if not text:
        continue

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=False,
    )["input_ids"]

    token_ids.extend(encoded)
    token_ids.append(tokenizer.eos_token_id)

    if len(token_ids) >= required_tokens:
        break

if len(token_ids) < required_tokens:
    raise RuntimeError(
        f"Required {required_tokens} tokens, but only "
        f"{len(token_ids)} were collected."
    )

blocks = [
    torch.tensor(
        token_ids[start:start + SEQ_LEN],
        dtype=torch.long,
    )
    for start in range(0, required_tokens, SEQ_LEN)
]

print("number of blocks:", len(blocks))
print("tokens per block:", SEQ_LEN)
print("total analyzed tokens:", len(blocks) * SEQ_LEN)


## 3. Reconstruct $Q$, $K$, $V$, and Attention Weights


In [ ]:
def split_heads(x, num_heads):
    # [batch, seq_len, hidden_size]
    # -> [batch, heads, seq_len, head_dim]
    batch_size, seq_len, hidden_size_local = x.shape
    local_head_dim = hidden_size_local // num_heads

    return (
        x.view(batch_size, seq_len, num_heads, local_head_dim)
        .permute(0, 2, 1, 3)
        .contiguous()
    )


def reconstruct_qkv_and_attention(
    model,
    attention_input,
    layer_index,
):
    # Reconstruct Q, K, V, and causal softmax attention weights
    # from the hidden states entering the attention module.
    attn = model.h[layer_index].attn

    qkv = attn.c_attn(attention_input)
    split_size = int(attn.split_size)
    query, key, value = qkv.split(split_size, dim=2)

    num_heads = int(attn.num_heads)
    query = split_heads(query, num_heads).float()
    key = split_heads(key, num_heads).float()
    value = split_heads(value, num_heads).float()

    scale = 1.0
    if bool(getattr(attn, "scale_attn_weights", True)):
        scale *= math.sqrt(query.shape[-1])
    if bool(getattr(attn, "scale_attn_by_inverse_layer_idx", False)):
        scale *= float(layer_index + 1)

    scores = torch.matmul(
        query,
        key.transpose(-1, -2),
    ) / scale

    seq_len = scores.shape[-1]
    causal_mask = torch.ones(
        (seq_len, seq_len),
        dtype=torch.bool,
        device=scores.device,
    ).tril()

    scores = scores.masked_fill(
        ~causal_mask.view(1, 1, seq_len, seq_len),
        torch.finfo(scores.dtype).min,
    )

    weights = torch.softmax(scores, dim=-1)

    return key, value, weights


## 4. Compute $C_\pi^i$ and Its Numerical Rank

For linear value aggregation, $\nabla\pi(v)=v$, so $C_\pi^i$ is computed for each head as an attention-weighted cross-covariance operator.

The numerical rank is computed with `torch.linalg.matrix_rank`. For this model, Corollary M.3 gives the bound $\min\{64,n_i-1\}$, where $n_i$ is the number of visible tokens.


In [ ]:
def analyze_layer(
    key,
    value,
    weights,
    block_index,
    layer_index,
):
    # key, value: [1, heads, seq_len, head_dim]
    # weights:    [1, heads, seq_len, seq_len]
    records = []

    key_dim = key.shape[-1]
    value_dim = value.shape[-1]

    for position in QUERY_POSITIONS:
        visible_tokens = position + 1

        current_weights = weights[0, :, position, :visible_tokens]
        current_key = key[0, :, :visible_tokens, :]
        current_value = value[0, :, :visible_tokens, :]

        current_weights = (
            current_weights
            / current_weights.sum(dim=-1, keepdim=True)
        )

        mean_key = torch.einsum(
            "hn,hnd->hd",
            current_weights,
            current_key,
        )
        mean_value = torch.einsum(
            "hn,hnd->hd",
            current_weights,
            current_value,
        )

        centered_key = current_key - mean_key[:, None, :]
        centered_value = current_value - mean_value[:, None, :]

        # C_pi^i: [heads, value_dim, key_dim]
        covariance = torch.einsum(
            "hn,hnd,hne->hde",
            current_weights,
            centered_value,
            centered_key,
        )

        # Numerical rank using the default float32 tolerance.
        numerical_rank = torch.linalg.matrix_rank(
            covariance.detach().cpu().float()
        )

        upper_bound = min(
            key_dim,
            value_dim,
            visible_tokens - 1,
        )

        for head_index in range(key.shape[1]):
            rank_value = int(numerical_rank[head_index].item())

            records.append(
                {
                    "block": block_index,
                    "layer": layer_index,
                    "head": head_index,
                    "query_position": position,
                    "visible_tokens": visible_tokens,
                    "upper_bound": upper_bound,
                    "numerical_rank": rank_value,
                    "attains_upper_bound": (
                        rank_value == upper_bound
                    ),
                    "rank_ratio": (
                        rank_value / upper_bound
                        if upper_bound > 0
                        else float("nan")
                    ),
                }
            )

    return records


## 5. Evaluate All Layers and Attention Heads


In [ ]:
all_records = []

with torch.inference_mode():
    for block_index, input_ids_1d in enumerate(
        tqdm(blocks, desc="analyzing text blocks")
    ):
        input_ids = input_ids_1d.unsqueeze(0).to(device)

        captured_attention_inputs.clear()

        _ = model(
            input_ids=input_ids,
            use_cache=False,
            return_dict=True,
        )

        if len(captured_attention_inputs) != n_layers:
            raise RuntimeError(
                "Failed to capture the attention input for every layer."
            )

        for layer_index in range(n_layers):
            key, value, weights = reconstruct_qkv_and_attention(
                model=model,
                attention_input=(
                    captured_attention_inputs[layer_index]
                ),
                layer_index=layer_index,
            )

            all_records.extend(
                analyze_layer(
                    key=key,
                    value=value,
                    weights=weights,
                    block_index=block_index,
                    layer_index=layer_index,
                )
            )

        del input_ids

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

for handle in hook_handles:
    handle.remove()

results = pd.DataFrame(all_records)

results_path = (
    RESULT_DIR / "gpt2_covariance_rank_results.csv"
)
results.to_csv(results_path, index=False)

print("number of observations:", len(results))
print("saved:", results_path)


## 6. Summarize Numerical Ranks by Visible Context Length

Aggregate the results for each number of visible tokens.


In [ ]:
summary = (
    results
    .groupby(
        ["query_position", "visible_tokens", "upper_bound"],
        as_index=False,
    )
    .agg(
        number_of_matrices=("numerical_rank", "size"),
        mean_rank=("numerical_rank", "mean"),
        median_rank=("numerical_rank", "median"),
        min_rank=("numerical_rank", "min"),
        max_rank=("numerical_rank", "max"),
        upper_bound_attainment_rate=(
            "attains_upper_bound",
            "mean",
        ),
        mean_rank_ratio=("rank_ratio", "mean"),
    )
)

summary_path = RESULT_DIR / "summary_by_position.csv"
summary.to_csv(summary_path, index=False)

display(summary)

print()
print("score-side potential: log-sum-exp")
print("value-side potential: quadratic")
print("saved:", summary_path)


## 7. Download Results


In [ ]:
archive_path = shutil.make_archive(
    "/content/gpt2_covariance",
    "zip",
    root_dir=RESULT_DIR,
)

print("created:", archive_path)

from google.colab import files
files.download(archive_path)
